In [1]:
import pandas as pd
!pip install tensorflow==2.21.0 keras==3.12.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 30.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0
  Attempting uninstall: keras
    Found existing installation: keras 3.13.2
    Uninstalling keras-3.13.2:
      Successfully uninstalled keras-3.13.2
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.20.0
    Uninstalling tensorflow-2.20.0:
      Successfully uninstalled tensorflow-2.20.0
ERROR: pip's dependency resolver does not currently 

In [2]:
data=pd.read_json('News_Category_Dataset_v3.json',lines=True)

In [3]:
data.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


# Data Exploration

In [4]:
data.shape

(209527, 6)

In [5]:
data.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date'], dtype='object')

In [6]:
data= data['short_description'].sample(20000)

In [7]:
data

,short_description
66208,
145821,While a housemaid attempts to hurry along the ...
124652,
55839,Sunday is Gold Star Mother's and Family's Day.
150749,There is the same gap when it comes to using p...
...,...
158428,"Whatever you are trying to achieve, I hope tha..."
52049,These two might be headed to the courtroom.
192923,CORRECTION: A previous version of this article...
18124,A former campaign staffer and a lobbyist have ...


# Data Cleaning

In [8]:
data.isnull().sum()

np.int64(0)

In [9]:
data.duplicated().sum()

np.int64(2023)

In [10]:
data.drop_duplicates(inplace=True)

# Tokenization


In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [12]:
tokenizer = Tokenizer(oov_token='<OOV')

In [13]:
tokenizer.fit_on_texts(data)

In [14]:
tokenizer.word_index

{'<OOV': 1,
 'the': 2,
 'to': 3,
 'a': 4,
 'of': 5,
 'and': 6,
 'in': 7,
 'is': 8,
 'that': 9,
 'for': 10,
 'i': 11,
 'you': 12,
 'on': 13,
 'it': 14,
 'are': 15,
 'with': 16,
 'as': 17,
 'be': 18,
 'we': 19,
 'this': 20,
 'have': 21,
 'was': 22,
 'but': 23,
 'at': 24,
 'not': 25,
 'your': 26,
 'from': 27,
 'my': 28,
 'has': 29,
 'an': 30,
 'or': 31,
 'more': 32,
 'about': 33,
 'our': 34,
 'by': 35,
 'one': 36,
 'can': 37,
 'what': 38,
 'all': 39,
 'when': 40,
 'their': 41,
 'will': 42,
 'they': 43,
 'his': 44,
 'who': 45,
 'out': 46,
 "it's": 47,
 'if': 48,
 'he': 49,
 'new': 50,
 'just': 51,
 'up': 52,
 'time': 53,
 'people': 54,
 'her': 55,
 'like': 56,
 'so': 57,
 'do': 58,
 'some': 59,
 'there': 60,
 'how': 61,
 'been': 62,
 'said': 63,
 'no': 64,
 'than': 65,
 'us': 66,
 'me': 67,
 'life': 68,
 'many': 69,
 'most': 70,
 'day': 71,
 'get': 72,
 'year': 73,
 'had': 74,
 'she': 75,
 'over': 76,
 'after': 77,
 'these': 78,
 'make': 79,
 'into': 80,
 'were': 81,
 'would': 82,
 'want':

In [15]:
len(tokenizer.word_index)

29160

In [16]:
data=tokenizer.texts_to_sequences(data)

In [17]:
input_sequences=[]
for seq in data:
    if len(seq)<2:
        continue
    for i in range(1,len(seq)):
        ngram=seq[:i+1]
        input_sequences.append(ngram)

In [18]:
input_sequences

[[100, 4],
 [100, 4, 15073],
 [100, 4, 15073, 2933],
 [100, 4, 15073, 2933, 3],
 [100, 4, 15073, 2933, 3, 10747],
 [100, 4, 15073, 2933, 3, 10747, 639],
 [100, 4, 15073, 2933, 3, 10747, 639, 2],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145, 539],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145, 539, 13],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145, 539, 13, 3],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145, 539, 13, 3, 108],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145, 539, 13, 3, 108, 4],
 [100, 4, 15073, 2933, 3, 10747, 639, 2, 315, 7145, 539, 13, 3, 108, 4, 1135],
 [100,
  4,
  15073,
  2933,
  3,
  10747,
  639,
  2,
  315,
  7145,
  539,
  13,
  3,
  108,
  4,
  1135,
  4905],
 [100,
  4,
  15073,
  2933,
  3,
  10747,
  639,
  2,
  315,
  7145,
  539,
  13,
  3,
  108,
  4,
  1135,
  4905,
  6],
 [100,
  4,
  15073,
  2933,
  3,
  10747,
  639,
  2,
  31

# Padding Implementation

In [19]:
max_len = max([len(i) for i in input_sequences])

In [20]:
max_len

185

In [21]:
input = pad_sequences(input_sequences, padding='pre',maxlen=max_len)

In [22]:
X=input[:,:-1]
y=input[:,-1]

In [23]:
X.shape

(376600, 184)

In [24]:
y.shape

(376600,)

In [25]:
y= np.array(y)

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [28]:
X_train.shape

(301280, 184)

In [29]:
vocab_size = len(tokenizer.word_index)+1

In [30]:
vocab_size

29161

In [31]:
max_len=X.shape[1]

In [32]:
model= Sequential([Embedding(vocab_size,128,input_length=max_len),
                   LSTM(256),
                   Dense(vocab_size, activation='softmax')])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [34]:
model.build(input_shape=(None,max_len))

In [35]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 184, 128)       │     3,732,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 29161)          │     7,494,377 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,621,225 (44.33 MB)

 Trainable params: 11,621,225 (44.33 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X_train,y_train, epochs=30,validation_data=(X_test,y_test))

Epoch 1/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 255s 26ms/step - accuracy: 0.0891 - loss: 7.1387 - val_accuracy: 0.1165 - val_loss: 6.7745
Epoch 2/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 229s 24ms/step - accuracy: 0.1276 - loss: 6.3641 - val_accuracy: 0.1295 - val_loss: 6.6739
Epoch 3/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 229s 24ms/step - accuracy: 0.1496 - loss: 5.8919 - val_accuracy: 0.1351 - val_loss: 6.7137
Epoch 4/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 262s 24ms/step - accuracy: 0.1709 - loss: 5.4197 - val_accuracy: 0.1333 - val_loss: 6.8346
Epoch 5/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 228s 24ms/step - accuracy: 0.1980 - loss: 4.9285 - val_accuracy: 0.1328 - val_loss: 7.0023
Epoch 6/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 228s 24ms/step - accuracy: 0.2341 - loss: 4.4565 - val_accuracy: 0.1282 - val_loss: 7.1972
Epoch 7/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 228s 24ms/step - accuracy: 0.2782 - loss: 4.0282 - val_accuracy: 0.1222 - val_loss: 7.4183
Epoch 8/30
9415/9415 ━━━━━━━━━━━━━━━━━━━━ 279s 26ms/step - accuracy: 

In [39]:
model.save('nextword_model.h5')

In [40]:
import pickle
with open ('tokenizer.pickle','wb') as file:
    pickle.dump(tokenizer,file)